# Bloque 2 - NLP: EDA y Preprocesamiento
## Dataset: Resume (Jarvis Calling Hiring Contest)
**Proyecto:** Segmentación de células e interpretación de texto mediante modelos de deep learning  
**Autor:** Dr. Lihki Rubio


## 0. Importaciones y configuración


In [1]:
import pandas as pd
import numpy as np
import re
import warnings
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams

from wordcloud import WordCloud
import base64
from io import BytesIO
import matplotlib.pyplot as plt  # solo para WordCloud → imagen base64

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Descargar recursos de NLTK necesarios
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


---
## 1. Carga de datos
Se carga el dataset con `pandas` tal como indica el PDF (sección 11.20.6.1.1.1).
El notebook vive en `notebooks/` — las rutas usan `../` para subir un nivel.


In [2]:
import os

# Ruta relativa desde notebooks/ hacia la raíz del proyecto
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH = os.path.join(BASE_DIR, 'data', 'Resume.csv')

df = pd.read_csv(DATA_PATH)

print(f'Shape del dataset: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
df.head()


Shape del dataset: (2484, 4)
Columnas: ['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
# Información general del dataframe
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [4]:
# Valores nulos
print('=== Valores nulos por columna ===')
print(df.isnull().sum())
print(f'\nTotal de filas duplicadas: {df.duplicated().sum()}')


=== Valores nulos por columna ===
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

Total de filas duplicadas: 0


In [5]:
# Identificar columnas de texto y etiqueta
TEXT_COL  = 'Resume_str'
LABEL_COL = 'Category'

assert TEXT_COL  in df.columns, f"Columna '{TEXT_COL}' no encontrada. Columnas: {df.columns.tolist()}"
assert LABEL_COL in df.columns, f"Columna '{LABEL_COL}' no encontrada. Columnas: {df.columns.tolist()}"

print(f'Columna de texto : {TEXT_COL}')
print(f'Columna de clase : {LABEL_COL}')
print(f'Número de clases : {df[LABEL_COL].nunique()}')


Columna de texto : Resume_str
Columna de clase : Category
Número de clases : 24


---
## 2. Análisis de clases
Sección 11.20.6.1.1.2: Conteo de clases con barplot.


In [6]:
class_counts = df[LABEL_COL].value_counts().reset_index()
class_counts.columns = ['Category', 'count']

print('=== Distribución de clases ===')
print(class_counts.to_string(index=False))
print(f'\nClase más frecuente : {class_counts.iloc[0]["Category"]} ({class_counts.iloc[0]["count"]} muestras)')
print(f'Clase menos frecuente: {class_counts.iloc[-1]["Category"]} ({class_counts.iloc[-1]["count"]} muestras)')
print(f'Ratio desbalance     : {class_counts["count"].max() / class_counts["count"].min():.2f}x')


=== Distribución de clases ===
              Category  count
INFORMATION-TECHNOLOGY    120
  BUSINESS-DEVELOPMENT    120
              ADVOCATE    118
                  CHEF    118
           ENGINEERING    118
            ACCOUNTANT    118
               FINANCE    118
               FITNESS    117
              AVIATION    117
                 SALES    116
               BANKING    115
            HEALTHCARE    115
            CONSULTANT    115
          CONSTRUCTION    112
      PUBLIC-RELATIONS    111
                    HR    110
              DESIGNER    107
                  ARTS    103
               TEACHER    102
               APPAREL     97
         DIGITAL-MEDIA     96
           AGRICULTURE     63
            AUTOMOBILE     36
                   BPO     22

Clase más frecuente : INFORMATION-TECHNOLOGY (120 muestras)
Clase menos frecuente: BPO (22 muestras)
Ratio desbalance     : 5.45x


In [7]:
mean_count = class_counts['count'].mean()

fig = px.bar(
    class_counts.sort_values('count'),
    x='count',
    y='Category',
    orientation='h',
    color='count',
    color_continuous_scale='Blues',
    title='Distribución de clases en el dataset Resume',
    labels={'count': 'Número de muestras', 'Category': 'Categoría'},
    text='count',
)
fig.add_vline(
    x=mean_count,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Media={mean_count:.0f}',
    annotation_position='top right',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    height=650,
    coloraxis_showscale=False,
    title_font_size=16,
)
fig.show()


---
## 3. Longitud de textos
Sección 11.20.6.1.1.3: Número de palabras por texto → Histograma + Boxplot.


In [8]:
df['word_count'] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))
df['char_count'] = df[TEXT_COL].astype(str).apply(len)

print('=== Estadísticas de longitud de textos (palabras) ===')
print(df['word_count'].describe().round(2))


=== Estadísticas de longitud de textos (palabras) ===
count    2484.00
mean      811.33
std       371.01
min         0.00
25%       651.00
50%       757.00
75%       933.00
max      5190.00
Name: word_count, dtype: float64


In [9]:
# Histograma de longitud
fig_hist = px.histogram(
    df,
    x='word_count',
    nbins=50,
    title='Histograma de longitud de textos (palabras)',
    labels={'word_count': 'Número de palabras', 'count': 'Frecuencia'},
    color_discrete_sequence=['steelblue'],
    opacity=0.85,
)
fig_hist.add_vline(
    x=df['word_count'].mean(),
    line_dash='dash', line_color='red',
    annotation_text=f'Media={df["word_count"].mean():.0f}',
    annotation_position='top right',
)
fig_hist.add_vline(
    x=df['word_count'].median(),
    line_dash='dash', line_color='orange',
    annotation_text=f'Mediana={df["word_count"].median():.0f}',
    annotation_position='top left',
)
fig_hist.update_layout(height=400, title_font_size=15)
fig_hist.show()


In [10]:
# Boxplot por categoría — ordenado por mediana descendente
category_order = (
    df.groupby(LABEL_COL)['word_count']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig_box = px.box(
    df,
    x='word_count',
    y=LABEL_COL,
    category_orders={LABEL_COL: category_order},
    title='Boxplot de longitud de textos por categoría',
    labels={'word_count': 'Número de palabras', LABEL_COL: 'Categoría'},
    color=LABEL_COL,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig_box.update_layout(
    height=750,
    showlegend=False,
    title_font_size=15,
)
fig_box.show()


---
## 4. Frecuencia de palabras
Sección 11.20.6.1.1.4: Top palabras → Barplot top 20 + WordCloud.


In [11]:
STOPWORDS_EN = set(stopwords.words('english'))

def tokenize_simple(text: str) -> list[str]:
    """Tokenización simple: minúsculas + solo alfanumérico + sin stopwords."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z]{3,}\b', text)
    tokens = [t for t in tokens if t not in STOPWORDS_EN]
    return tokens

all_tokens = []
for text in df[TEXT_COL].astype(str):
    all_tokens.extend(tokenize_simple(text))

word_freq = Counter(all_tokens)
top20 = word_freq.most_common(20)

print(f'Vocabulario total (sin stopwords): {len(word_freq):,} tokens únicos')
print('\nTop 20 palabras:')
for word, count in top20:
    print(f'  {word:<20} {count:>6}')


Vocabulario total (sin stopwords): 36,895 tokens únicos

Top 20 palabras:
  state                 16159
  company               15212
  city                  15077
  management            12173
  name                  11739
  sales                  8275
  customer               7927
  business               7866
  skills                 7729
  new                    6407
  service                6223
  team                   6073
  development            5731
  training               5637
  experience             5631
  project                5362
  work                   4817
  manager                4534
  information            4502
  marketing              4489


In [12]:
# Barplot top 20
words_top20  = [w for w, _ in top20]
counts_top20 = [c for _, c in top20]

fig_top20 = px.bar(
    x=counts_top20[::-1],
    y=words_top20[::-1],
    orientation='h',
    title='Top 20 palabras más frecuentes (sin stopwords)',
    labels={'x': 'Frecuencia', 'y': 'Palabra'},
    color=counts_top20[::-1],
    color_continuous_scale='Blues',
    text=counts_top20[::-1],
)
fig_top20.update_traces(textposition='outside')
fig_top20.update_layout(
    height=550,
    coloraxis_showscale=False,
    title_font_size=15,
)
fig_top20.show()


In [13]:
# WordCloud — se genera con matplotlib y se incrusta como imagen en Plotly
wc = WordCloud(
    width=900, height=450,
    background_color='white',
    max_words=150,
    colormap='Blues',
    random_state=RANDOM_SEED,
).generate_from_frequencies(word_freq)

# Convertir a imagen base64 para incrustar en Plotly
buf = BytesIO()
fig_wc_mpl, ax = plt.subplots(figsize=(11, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
fig_wc_mpl.tight_layout(pad=0)
fig_wc_mpl.savefig(buf, format='png', dpi=150, bbox_inches='tight')
plt.close(fig_wc_mpl)
buf.seek(0)
img_b64 = base64.b64encode(buf.read()).decode('utf-8')

fig_wc = go.Figure()
fig_wc.add_layout_image(
    dict(
        source=f'data:image/png;base64,{img_b64}',
        xref='paper', yref='paper',
        x=0, y=1,
        sizex=1, sizey=1,
        xanchor='left', yanchor='top',
        layer='below',
    )
)
fig_wc.update_layout(
    title='WordCloud del corpus completo',
    title_font_size=15,
    height=450,
    margin=dict(l=0, r=0, t=40, b=0),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)
fig_wc.show()


---
## 5. N-gramas
Sección 11.20.6.1.1.5: Bigramas y trigramas más frecuentes.


In [14]:
def get_ngrams(token_list: list[str], n: int) -> Counter:
    """Genera n-gramas y retorna su frecuencia."""
    ngram_list = list(ngrams(token_list, n))
    return Counter([' '.join(g) for g in ngram_list])

bigram_freq  = get_ngrams(all_tokens, 2)
trigram_freq = get_ngrams(all_tokens, 3)

top15_bi  = bigram_freq.most_common(15)
top15_tri = trigram_freq.most_common(15)

print('=== Top 15 Bigramas ===')
for ng, cnt in top15_bi:
    print(f'  {ng:<35} {cnt:>5}')

print('\n=== Top 15 Trigramas ===')
for ng, cnt in top15_tri:
    print(f'  {ng:<45} {cnt:>5}')


=== Top 15 Bigramas ===
  city state                          14107
  company name                        11563
  name city                            9900
  customer service                     3061
  project management                   1198
  current company                      1177
  microsoft office                     1147
  university city                      1107
  high school                          1069
  business development                  914
  communication skills                  842
  human resources                       819
  public relations                      783
  social media                          760
  problem solving                       759

=== Top 15 Trigramas ===
  company name city                              9900
  name city state                                9719
  current company name                           1173
  university city state                          1030
  manager company name                            704
  january company na

In [15]:
# Bigramas
bi_phrases = [w for w, _ in top15_bi]
bi_counts  = [c for _, c in top15_bi]

fig_bi = px.bar(
    x=bi_counts[::-1],
    y=bi_phrases[::-1],
    orientation='h',
    title='Top 15 Bigramas más frecuentes',
    labels={'x': 'Frecuencia', 'y': 'Bigrama'},
    color=bi_counts[::-1],
    color_continuous_scale='Greens',
    text=bi_counts[::-1],
)
fig_bi.update_traces(textposition='outside')
fig_bi.update_layout(height=500, coloraxis_showscale=False, title_font_size=15)
fig_bi.show()


In [16]:
# Trigramas
tri_phrases = [w for w, _ in top15_tri]
tri_counts  = [c for _, c in top15_tri]

fig_tri = px.bar(
    x=tri_counts[::-1],
    y=tri_phrases[::-1],
    orientation='h',
    title='Top 15 Trigramas más frecuentes',
    labels={'x': 'Frecuencia', 'y': 'Trigrama'},
    color=tri_counts[::-1],
    color_continuous_scale='Oranges',
    text=tri_counts[::-1],
)
fig_tri.update_traces(textposition='outside')
fig_tri.update_layout(height=500, coloraxis_showscale=False, title_font_size=15)
fig_tri.show()


---
## 6. Análisis de ruido y calidad de texto


In [17]:
df['has_url']     = df[TEXT_COL].astype(str).str.contains(r'http[s]?://', regex=True)
df['has_email']   = df[TEXT_COL].astype(str).str.contains(r'\S+@\S+', regex=True)
df['has_numbers'] = df[TEXT_COL].astype(str).str.contains(r'\d+', regex=True)
df['has_special'] = df[TEXT_COL].astype(str).str.contains(r'[^\w\s]', regex=True)

noise_summary = pd.DataFrame({
    'Tipo de ruido'    : ['URLs', 'Emails', 'Números', 'Caracteres especiales'],
    'Textos afectados' : [
        df['has_url'].sum(),
        df['has_email'].sum(),
        df['has_numbers'].sum(),
        df['has_special'].sum(),
    ],
})
noise_summary['Porcentaje (%)'] = (noise_summary['Textos afectados'] / len(df) * 100).round(2)
print(noise_summary.to_string(index=False))


        Tipo de ruido  Textos afectados  Porcentaje (%)
                 URLs                58            2.33
               Emails                25            1.01
              Números              2483           99.96
Caracteres especiales              2483           99.96


In [18]:
fig_noise = px.bar(
    noise_summary,
    x='Tipo de ruido',
    y='Porcentaje (%)',
    title='Presencia de ruido en los textos',
    color='Porcentaje (%)',
    color_continuous_scale='Reds',
    text='Porcentaje (%)',
    labels={'Porcentaje (%)': 'Porcentaje de textos afectados (%)'},
)
fig_noise.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_noise.update_layout(height=400, coloraxis_showscale=False, title_font_size=15)
fig_noise.show()


---
## 7. Interpretación del EDA
Sección 11.20.6.1.2


In [19]:
balance_ratio = class_counts['count'].max() / class_counts['count'].min()
mean_words    = df['word_count'].mean()
std_words     = df['word_count'].std()
cv_words      = std_words / mean_words

print('======================================================')
print('         RESUMEN INTERPRETATIVO DEL EDA')
print('======================================================')
print(f'\n1. Balance de clases')
print(f'   Ratio max/min           : {balance_ratio:.2f}x')
print(f'   ¿Clases balanceadas?    : {"SÍ" if balance_ratio < 1.5 else "NO - hay desbalance"}')

print(f'\n2. Longitud de textos')
print(f'   Media palabras          : {mean_words:.0f}')
print(f'   Desviación estándar     : {std_words:.0f}')
print(f'   Coef. de variación      : {cv_words:.2f}')
print(f'   ¿Textos cortos o largos?: {"LARGOS" if mean_words > 200 else "CORTOS"}')
print(f'   ¿Alta variabilidad?     : {"SÍ" if cv_words > 0.5 else "NO"}')

print(f'\n3. Ruido en los textos')
pct_url   = df['has_url'].mean()   * 100
pct_email = df['has_email'].mean() * 100
pct_spec  = df['has_special'].mean() * 100
print(f'   Textos con URLs         : {pct_url:.1f}%')
print(f'   Textos con emails       : {pct_email:.1f}%')
print(f'   Textos con simbolos     : {pct_spec:.1f}%')
print(f'   ¿Hay ruido notable?     : {"SÍ" if (pct_url + pct_email) > 10 else "MODERADO"}')
print('======================================================')


         RESUMEN INTERPRETATIVO DEL EDA

1. Balance de clases
   Ratio max/min           : 5.45x
   ¿Clases balanceadas?    : NO - hay desbalance

2. Longitud de textos
   Media palabras          : 811
   Desviación estándar     : 371
   Coef. de variación      : 0.46
   ¿Textos cortos o largos?: LARGOS
   ¿Alta variabilidad?     : NO

3. Ruido en los textos
   Textos con URLs         : 2.3%
   Textos con emails       : 1.0%
   Textos con simbolos     : 100.0%
   ¿Hay ruido notable?     : MODERADO


---
## 8. Preprocesamiento
Sección 11.20.6.2: Pasos obligatorios según el PDF.
1. Minúsculas
2. Eliminación de símbolos
3. Eliminación de stopwords
4. Tokenización
5. Padding (se aplica después con Keras)


In [20]:
def preprocess_text(text: str, stopwords_set: set = STOPWORDS_EN) -> str:
    """
    Pipeline de preprocesamiento de texto (Sección 11.20.6.2):
      1. Conversión a minúsculas
      2. Eliminación de URLs y emails
      3. Eliminación de caracteres no alfabéticos
      4. Eliminación de stopwords
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in stopwords_set and len(t) > 2]
    return ' '.join(tokens)


print('Aplicando preprocesamiento...')
df['clean_text'] = df[TEXT_COL].apply(preprocess_text)
print('✓ Preprocesamiento completado.')

df['word_count_clean'] = df['clean_text'].apply(lambda x: len(x.split()))

print(f'\nLongitud media ANTES : {df["word_count"].mean():.0f} palabras')
print(f'Longitud media DESPUÉS: {df["word_count_clean"].mean():.0f} palabras')
print(f'Reducción            : {(1 - df["word_count_clean"].mean()/df["word_count"].mean())*100:.1f}%')


Aplicando preprocesamiento...
✓ Preprocesamiento completado.

Longitud media ANTES : 811 palabras
Longitud media DESPUÉS: 584 palabras
Reducción            : 28.0%


In [21]:
# Ejemplo de preprocesamiento
sample_idx = df.sample(1, random_state=RANDOM_SEED).index[0]

print('=== EJEMPLO DE PREPROCESAMIENTO ===')
print('\n--- TEXTO ORIGINAL (primeros 500 chars) ---')
print(df.loc[sample_idx, TEXT_COL][:500])
print('\n--- TEXTO LIMPIO (primeros 500 chars) ---')
print(df.loc[sample_idx, 'clean_text'][:500])
print(f'\nClase: {df.loc[sample_idx, LABEL_COL]}')


=== EJEMPLO DE PREPROCESAMIENTO ===

--- TEXTO ORIGINAL (primeros 500 chars) ---
           Kpandipou    Koffi         Summary      Compassionate teaching professional delivering exemplary support and assistance to teachers and students. Display exceptional Communication and problem solving skills.  Experience in office administration and public speaking. Attentive and adaptable, skilled in management of classroom operations. Effective in leveraging student feedback to create dynamic lesson plans that address individual strengths and weaknesses.  Dedicated and responsive tea

--- TEXTO LIMPIO (primeros 500 chars) ---
kpandipou koffi summary compassionate teaching professional delivering exemplary support assistance teachers students display exceptional communication problem solving skills experience office administration public speaking attentive adaptable skilled management classroom operations effective leveraging student feedback create dynamic lesson plans address individual streng

In [22]:
# Comparación de longitudes: antes vs después
fig_comp = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Texto ORIGINAL', 'Texto PREPROCESADO'],
)

fig_comp.add_trace(
    go.Histogram(x=df['word_count'], nbinsx=50, marker_color='steelblue',
                 opacity=0.85, name='Original'),
    row=1, col=1,
)
fig_comp.add_vline(
    x=df['word_count'].mean(), line_dash='dash', line_color='red',
    annotation_text=f'Media={df["word_count"].mean():.0f}',
    row=1, col=1,
)

fig_comp.add_trace(
    go.Histogram(x=df['word_count_clean'], nbinsx=50, marker_color='seagreen',
                 opacity=0.85, name='Preprocesado'),
    row=1, col=2,
)
fig_comp.add_vline(
    x=df['word_count_clean'].mean(), line_dash='dash', line_color='red',
    annotation_text=f'Media={df["word_count_clean"].mean():.0f}',
    row=1, col=2,
)

fig_comp.update_layout(
    title_text='Longitud de textos: antes vs después del preprocesamiento',
    title_font_size=15,
    height=420,
    showlegend=False,
)
fig_comp.update_xaxes(title_text='Número de palabras')
fig_comp.update_yaxes(title_text='Frecuencia')
fig_comp.show()


---
## 9. Tokenización con Keras y Padding
Sección 11.20.6.2 paso 4-5. Se determina `maxlen` basado en el EDA.


In [23]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

print(f'TensorFlow version: {tf.__version__}')

MAX_LEN_P95 = int(np.percentile(df['word_count_clean'], 95))
MAX_LEN     = 100 if MAX_LEN_P95 > 75 else 50

print(f'Percentil 95 de longitud limpia : {MAX_LEN_P95} palabras')
print(f'MAX_LEN seleccionado (EDA-based) : {MAX_LEN}')


2026-05-17 17:31:32.856157: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-17 17:31:33.597636: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-17 17:31:36.540611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
Percentil 95 de longitud limpia : 1008 palabras
MAX_LEN seleccionado (EDA-based) : 100


In [24]:
VOCAB_SIZE = 20_000

le = LabelEncoder()
y_encoded = le.fit_transform(df[LABEL_COL])

print(f'Clases ({len(le.classes_)}): {list(le.classes_)}')

X_text  = df['clean_text'].values
idx_all = np.arange(len(X_text))

idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx_all, y_encoded,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_encoded,
)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

print(f'\n=== Tamaños de splits ===')
print(f'  Train      : {len(idx_train):>5} muestras  ({len(idx_train)/len(idx_all)*100:.1f}%)')
print(f'  Validation : {len(idx_val):>5} muestras  ({len(idx_val)/len(idx_all)*100:.1f}%)')
print(f'  Test       : {len(idx_test):>5} muestras  ({len(idx_test)/len(idx_all)*100:.1f}%)')


Clases (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']

=== Tamaños de splits ===
  Train      :  1738 muestras  (70.0%)
  Validation :   373 muestras  (15.0%)
  Test       :   373 muestras  (15.0%)


In [25]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_text[idx_train])   # ← solo train

vocab_actual = len(tokenizer.word_index)
print(f'Vocabulario total encontrado : {vocab_actual:,} tokens')
print(f'Vocabulario limitado a       : {VOCAB_SIZE:,} tokens')
print(f'Cobertura aproximada         : {min(VOCAB_SIZE/vocab_actual*100, 100):.1f}%')

def encode_and_pad(texts, tokenizer, max_len):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

X_train = encode_and_pad(X_text[idx_train], tokenizer, MAX_LEN)
X_val   = encode_and_pad(X_text[idx_val],   tokenizer, MAX_LEN)
X_test  = encode_and_pad(X_text[idx_test],  tokenizer, MAX_LEN)

print(f'\nShapes resultantes:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')


Vocabulario total encontrado : 31,295 tokens
Vocabulario limitado a       : 20,000 tokens
Cobertura aproximada         : 63.9%

Shapes resultantes:
  X_train : (1738, 100)
  X_val   : (373, 100)
  X_test  : (373, 100)


---
## 10. Guardar artefactos del preprocesamiento


In [26]:
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

with open(os.path.join(ARTIFACTS_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

np.save(os.path.join(ARTIFACTS_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(ARTIFACTS_DIR, 'X_val.npy'),   X_val)
np.save(os.path.join(ARTIFACTS_DIR, 'X_test.npy'),  X_test)
np.save(os.path.join(ARTIFACTS_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(ARTIFACTS_DIR, 'y_val.npy'),   y_val)
np.save(os.path.join(ARTIFACTS_DIR, 'y_test.npy'),  y_test)

config = {
    'MAX_LEN'     : MAX_LEN,
    'VOCAB_SIZE'  : VOCAB_SIZE,
    'NUM_CLASSES' : len(le.classes_),
    'CLASSES'     : list(le.classes_),
    'TEXT_COL'    : TEXT_COL,
    'LABEL_COL'   : LABEL_COL,
}
with open(os.path.join(ARTIFACTS_DIR, 'config.pkl'), 'wb') as f:
    pickle.dump(config, f)

print('✓ Artefactos guardados en artifacts/')
print('  - tokenizer.pkl   (fiteado solo sobre train)')
print('  - label_encoder.pkl')
print('  - X_train/val/test.npy')
print('  - y_train/val/test.npy')
print('  - config.pkl')
print(f'\nConfiguración del pipeline:')
for k, v in config.items():
    if k != 'CLASSES':
        print(f'  {k:<15}: {v}')


✓ Artefactos guardados en artifacts/
  - tokenizer.pkl   (fiteado solo sobre train)
  - label_encoder.pkl
  - X_train/val/test.npy
  - y_train/val/test.npy
  - config.pkl

Configuración del pipeline:
  MAX_LEN        : 100
  VOCAB_SIZE     : 20000
  NUM_CLASSES    : 24
  TEXT_COL       : Resume_str
  LABEL_COL      : Category


---
## Resumen final

| Paso | Estado |
|---|---|
| Carga de datos (pandas) | ✅ |
| Análisis de clases (barplot) | ✅ |
| Longitud de textos (histograma + boxplot) | ✅ |
| Frecuencia de palabras (barplot + WordCloud) | ✅ |
| N-gramas (bigramas y trigramas) | ✅ |
| Análisis de ruido | ✅ |
| Preprocesamiento (minúsculas + símbolos + stopwords) | ✅ |
| Tokenización (Keras) | ✅ |
| Padding (Keras) | ✅ |
| Split Train/Val/Test 70/15/15 | ✅ |
| Guardado de artefactos | ✅ |

**Siguientes pasos:** Notebook `02_Model_TFIDF_XGBoost.ipynb` → TF-IDF + XGBoost.
